# 05 — Baseline Models

**Project:** Healthcare Readmission Intelligence  
**Input:** `data/features/train_features.csv` and `data/features/test_features.csv`  
**Output:** Trained baseline models + evaluation report

## Objective

Before building the Random Forest, we need a baseline — a simple model that tells us
what "good enough to beat" looks like. Without a baseline, we have no way to know if
our Random Forest is actually adding value.

We will train two baselines:

| Model | Why |
| ----- | --- |
| Dummy Classifier | The floor — what a model that ignores all features gets |
| Logistic Regression | A strong linear baseline — fast, interpretable, well-understood |

## Evaluation Strategy

We use these metrics — **not accuracy** — because the dataset is 88.7 / 11.3 imbalanced:

| Metric | Why it matters here |
| ------ | ------------------- |
| ROC-AUC | Overall discrimination ability regardless of threshold |
| F1 (class 1) | Balance of precision and recall for the minority class |
| Recall (class 1) | How many actual readmissions we catch — clinically critical |
| Precision (class 1) | Of the ones we flag, how many are actually readmitted |

> A model that predicts 0 for everything gets 88.7% accuracy but 0.0 F1 and 0.5 AUC.
> That is our true floor — the Dummy Classifier will confirm this.

In [1]:
import warnings
import os
import sys
import yaml

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder, StandardScaler
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

warnings.filterwarnings("ignore")
plt.style.use("ggplot")
pd.set_option("display.max_columns", None)

In [2]:
CONFIG_PATH = '../configs/paths.yaml'

with open(CONFIG_PATH,"r") as file:
    paths = yaml.safe_load(file)
df_train = pd.read_csv(paths['features_data']['train_data'])
df_test = pd.read_csv(paths['features_data']['test_data'])

df_train.head()

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,acarbose,insulin,glyburide-metformin,glipizide-metformin,change,diabetesMed,diag_1_group,diag_2_group,diag_3_group,medical_specialty_group,total_prior_visits,high_utilizer,readmitted_binary
0,Caucasian,Female,0,6,25,1,1,Unknown,41,0,1,0,0,0,1,No,No,No,No,No,No,No,No,No,No,0,No,No,No,No,Diabetes,Unknown,Unknown,Pediatrics,0,0,0
1,Caucasian,Female,1,1,1,7,3,Unknown,59,0,18,0,0,0,9,No,No,No,No,No,No,No,No,No,No,3,No,No,Yes,Yes,Other,Diabetes,Other,Missing,0,0,0
2,AfricanAmerican,Female,2,1,1,7,2,Unknown,11,5,13,2,0,1,6,No,No,No,No,No,Yes,No,No,No,No,0,No,No,No,Yes,Other,Diabetes,Other,Missing,3,1,0
3,Caucasian,Male,3,1,1,7,2,Unknown,44,1,16,0,0,0,7,No,No,No,No,No,No,No,No,No,No,3,No,No,Yes,Yes,Other,Diabetes,Circulatory,Missing,0,0,0
4,Caucasian,Male,5,2,1,2,3,Unknown,31,6,16,0,0,0,9,No,No,No,No,No,No,No,No,No,No,1,No,No,No,Yes,Circulatory,Circulatory,Diabetes,Missing,0,0,0


In [3]:
X_train = df_train.drop(columns = ['readmitted_binary'])
X_test = df_test.drop(columns = ['readmitted_binary'])

y_train = df_train['readmitted_binary']
y_test = df_test['readmitted_binary']


# Preprocessor and Helper functions

In [ ]:
def evaluate_classifier(name,pipeline,X_train,X_test,y_train,y_test):
    pipeline.fit(X_train,y_train)
    y_pred =  pipeline.predict(y_train)

In [4]:
cols_to_encode = [
    'race', 
    'payer_code', 
    'diag_1_group', 
    'diag_2_group', 
    'diag_3_group', 
    'medical_specialty_group',
    'admission_type_id',  
    'discharge_disposition_id', 
    'admission_source_id',
    'gender', 
    'change', 
    'diabetesMed',
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 
    'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 
    'rosiglitazone', 'acarbose', 'glyburide-metformin', 'glipizide-metformin'
]

cols_to_scale = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses',
    'total_prior_visits'
] 

In [5]:
preprocessor =  ColumnTransformer(
    transformers = [
        (
            'one_hot_encoder',
            OneHotEncoder(
                drop='first',
                handle_unknown='ignore',
                sparse_output = False
            ),
            cols_to_encode
        ),
        (
            'scaler',
            StandardScaler(),
            cols_to_scale
        )
    ],
    remainder='passthrough'
)

In [6]:
baseline_models = {
    'Dummy (most_frequent)': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression':   LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=30
    )
}

In [7]:
for name,model in baselilne_models.items():
    pipeline = Pipeline(
        ('preprocessor',preprocessor),
        (name,model)
    )

    

NameError: name 'baselilne_models' is not defined